# Requirement
- create a df with the following fields:
  - races:
    - race_year
    - race_name
    - race_date
  - circuits:
    - circuit_location
  - drivers:
    - driver_name
    - driver_number
    - driver_nationality
  - constructors: 
    - team
  - results: 
    - grid
    - fastests_lap
    - race_time
    - points
    - created_date
  

In [0]:
%run "../includes/configuration"

In [0]:
from pyspark.sql.functions import lit, current_timestamp

In [0]:
results_df = spark.read.parquet(f'{processed_folder_path}/results').withColumnRenamed('time', 'race_time')

In [0]:
races_df = spark.read.parquet(f'{processed_folder_path}/races').withColumnRenamed('name', 'race_name')

In [0]:
drivers_df = spark.read.parquet(f'{processed_folder_path}/drivers').withColumnRenamed('name', 'driver_name').withColumnRenamed('nationality', 'driver_nationality').withColumnRenamed('number', 'driver_number')

In [0]:
constructors_df = spark.read.parquet(f'{processed_folder_path}/constructors').withColumnRenamed('name', 'team')

In [0]:
circuits_df = spark.read.parquet(f'{processed_folder_path}/circuits').withColumnRenamed('location', 'circuit_location')

In [0]:
#display(results_df)
#display(races_df)
#display(drivers_df)
#display(constructors_df)

In [0]:
final_report_df= races_df.join(results_df, races_df.race_id == results_df.race_id, 'inner').join(drivers_df, results_df.driver_id == drivers_df.driver_id, 'inner').join(constructors_df, results_df.constructor_id == constructors_df.constructor_id, 'inner').join(circuits_df, races_df.circuit_id == circuits_df.circuit_id, 'inner').select('race_year', 'race_name', 'race_timestamp', 'circuit_location', 'driver_name', 'driver_number', 'driver_nationality', 'team','grid','fastest_lap_time','race_time','points').withColumn('created_date',lit(current_timestamp()))
#display(final_report_df)


In [0]:
final_report_df.write.mode('overwrite').parquet(f'{presentation_folder_path}/race_results')